In [ ]:
# ==== SETUP ====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib

# Load lại data đã split từ notebook trước (bản gốc, chưa qua SMOTE)
X_train, X_test, y_train, y_test = joblib.load('data/processed_split.pkl')
print("Tỷ lệ fraud trong y_test:", y_test.mean())  # sanity check: phải là tỷ lệ gốc, rất nhỏ

# Load cả 2 model ứng viên từ tuần 3 + bảng metrics để so sánh
xgb_model = joblib.load('results/xgboost_model.pkl')
xgb_smote_model = joblib.load('results/xgboost_smote_model.pkl')
metrics_df = joblib.load('results/xgboost_comparison_metrics.pkl')
print(metrics_df)

# Chọn model tốt nhất dựa trên Recall/PR-AUC thực tế (sửa dòng dưới theo số liệu in ra ở trên)
best_model = xgb_smote_model  # hoặc xgb_model — tùy metrics_df cho thấy cái nào thắng
best_model_name = "XGBoost + SMOTE"  # dùng cho title các plot sau này

# Sanity check cuối
print("Model được chọn:", best_model_name, "-", type(best_model).__name__)
print("X_test shape:", X_test.shape)

In [ ]:
# Với XGBoost, dùng TreeExplainer (nhanh, chính xác cho tree-based model)
explainer = shap.TreeExplainer(best_model)

# Tính SHAP values trên tập test
X_test_sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

print("SHAP values shape:", shap_values.shape)

In [ ]:
shap.summary_plot(shap_values, X_test_sample, show=False)
plt.tight_layout()
plt.savefig('results/shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tìm 1 True Positive (fraud, model đoán đúng) và 1 False Negative (fraud, model bỏ sót)
y_pred_sample = best_model.predict(X_test_sample)
y_test_sample = y_test.loc[X_test_sample.index]

tp_idx = X_test_sample.index[(y_test_sample == 1) & (y_pred_sample == 1)][0]
fn_idx = X_test_sample.index[(y_test_sample == 1) & (y_pred_sample == 0)][0]

print("True Positive case index:", tp_idx)
print("False Negative case index:", fn_idx)



# Fallback: nếu sample 5000 dòng không có đủ case, tìm riêng trong toàn bộ y_test == 1
if fn_mask.sum() == 0:
    print("Không tìm thấy False Negative trong sample — tìm riêng trong toàn bộ fraud cases...")
    fraud_idx_all = y_test[y_test == 1].index
    y_pred_all_fraud = best_model.predict(X_test.loc[fraud_idx_all])
    fn_idx_candidates = fraud_idx_all[y_pred_all_fraud == 0]
    if len(fn_idx_candidates) == 0:
        print("Model không bỏ sót fraud case nào trên toàn bộ test set (Recall = 100%).")
        fn_idx = None
    else:
        fn_idx = fn_idx_candidates[0]
        # Thêm case này vào X_test_sample + tính lại SHAP value riêng cho nó
        if fn_idx not in X_test_sample.index:
            X_test_sample = pd.concat([X_test_sample, X_test.loc[[fn_idx]]])
            shap_values_fn_extra = explainer.shap_values(X_test.loc[[fn_idx]])
else:
    fn_idx = X_test_sample.index[fn_mask][0]

if tp_mask.sum() == 0:
    print("Không tìm thấy True Positive trong sample — kiểm tra lại model, đây là trường hợp bất thường.")
    tp_idx = None
else:
    tp_idx = X_test_sample.index[tp_mask][0]

print("True Positive case index:", tp_idx)
print("False Negative case index:", fn_idx)

In [ ]:
tp_position = X_test_sample.index.get_loc(tp_idx)

shap.force_plot(
    explainer.expected_value,
    shap_values[tp_position],
    X_test_sample.iloc[tp_position],
    matplotlib=True,
    show=False
)
plt.savefig('results/shap_force_plot_TP.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if fn_idx is None:
    print("Model không có False Negative nào trên toàn bộ test set — không có case để vẽ force plot.")
    print("Đây là điểm đáng ghi nhận: Recall = 100% trên test set, có thể nêu trong report.")
else:
    # Trường hợp fn_idx nằm trong X_test_sample gốc/đã mở rộng ở Cell 5
    fn_position = X_test_sample.index.get_loc(fn_idx)

    # Nếu fn_idx là case được thêm riêng ở phần fallback (không có sẵn shap_values cho nó
    # trong mảng shap_values gốc), dùng shap_values_fn_extra thay vì shap_values
    if 'shap_values_fn_extra' in dir() and fn_position == len(X_test_sample) - 1 and fn_position >= len(shap_values):
        fn_shap_value = shap_values_fn_extra[0]
    else:
        fn_shap_value = shap_values[fn_position]

    shap.force_plot(
        explainer.expected_value,
        fn_shap_value,
        X_test_sample.iloc[fn_position],
        matplotlib=True,
        show=False
    )
    plt.savefig('results/shap_force_plot_FN.png', dpi=150, bbox_inches='tight')
    plt.show()